# Python. Семинар 1. Типы данных вблизи


## Как сдаются домашние задания

При регистрации на курсе в [manytask](https://hsemanytask.org) создаётся личный
репозиторий-форк. Новые наборы задач приходят в него обновлением из публичного
репозитория курса. Порядок работы такой:

1. забрать обновление и решить задачу у себя;
2. `git push` в свой репозиторий;
3. автоматическая проверка прогоняет тесты и выставляет балл в manytask.

Полная инструкция — установка Python, настройка доступа, клонирование — лежит в
`README.md` репозитория курса.

**Тесты.** Рядом с задачей лежит `test_public.py` — эти тесты можно и нужно
запускать локально до отправки. Балл считается по второму, закрытому набору
тестов, который виден только проверяющей системе: он проверяет те же функции на
дополнительных случаях. Если публичные тесты не прошли, закрытые не запускаются
и балл будет нулевым.

**Баллы.** Задачи внутри набора стоят разное число баллов в зависимости от
сложности. manytask показывает, сколько баллов набрано, какой это процент от
максимума и какая оценка по десятибалльной шкале этому проценту соответствует.

**Дедлайна у набора два, и оба видны в manytask:**

| Срок | Что происходит |
|---|---|
| мягкий — через неделю после лекции | после него задача оценивается вполовину |
| жёсткий — через две недели после лекции | после него задача не оценивается вовсе |

Так устроены наборы к лекциям. У вводной задачи `hello_world`, которая выдаётся
до начала занятий, сроки короче — точные даты, как и для всего остального,
стоят в manytask.

**Число попыток не ограничено.** Штрафа за количество отправок нет, и балл за
задачу никогда не уменьшается: засчитывается лучший результат. Отправлять
недоделанное решение, чтобы проверить себя, совершенно нормально.

**Задачи семинаров баллов не дают.** Те, что вы будете решать сегодня, лежат в
каталоге `seminar` рядом с лекцией: они проверяются очно на паре и в
автоматическую проверку не попадают.


## Окружение проекта

Библиотеки ставятся не «в Python вообще», а в конкретное **окружение**. Своё
окружение на проект снимает конфликт версий: одному проекту нужна библиотека
версии 1.1, другому — 2.0, а установлена может быть только одна.

Создаётся и включается один раз на проект:

```bash
python3 -m venv .venv
source .venv/bin/activate
```

Приглашение терминала получит префикс `(.venv)`: теперь `python` и `pip`
указывают на окружение, а не на системный интерпретатор. Проверить, куда именно:

```bash
which python      # …/ваш-проект/.venv/bin/python
pip -V            # тот же путь
```

Активация действует в одном окне терминала — в новом её повторяют. Папку `.venv`
в git не коммитят: она весит сотни мегабайт и восстанавливается одной командой,
поэтому уже вписана в `.gitignore`.

> **Типичная ошибка.** `ModuleNotFoundError` сразу после успешного
> `pip install`. Значит, ставили в одно окружение, а запускаете другим
> интерпретатором. Сверьте `pip -V` и `which python` — оба пути должны вести в
> одну и ту же папку `.venv`.

## 1. `bool` — это число

В лекции `bool` был отдельным типом со значениями `True` и `False`. На самом
деле `bool` — это подтип `int`: `True` равен единице, `False` равен нулю.


In [ ]:
isinstance(True, int), True == 1, False == 0

In [ ]:
True + True + True

Отсюда полезный приём: результат сравнения можно использовать в арифметике.
Например, знак числа — это разность двух сравнений, без всяких условий:


In [ ]:
n = -3
(n > 0) - (n < 0)

In [ ]:
(5 > 0) - (5 < 0), (0 > 0) - (0 < 0)

Так же проверяется совпадение знаков: `(a < 0) == (b < 0)`. Условных
конструкций мы ещё не проходили — арифметика над сравнениями их заменяет.


## 2. Целые числа: почему они не переполняются

В большинстве языков целое число — это фиксированные 32 или 64 бита, и при
выходе за границу происходит переполнение. В Python `int` растёт, пока хватает
памяти.


In [ ]:
2**1000

Устроено это так: число хранится как последовательность «цифр» по 30 бит
каждая. Сколько бит в цифре — можно спросить у интерпретатора:


In [ ]:
import sys

sys.int_info.bits_per_digit

`sys.getsizeof` показывает, сколько байт занимает объект. Пока число помещается
в одну 30-битную цифру, размер не меняется; на числе `2 ** 30` добавляется
вторая цифра, и объект становится больше:


In [ ]:
sys.getsizeof(0), sys.getsizeof(2**29), sys.getsizeof(2**30), sys.getsizeof(2**1000)

Плата за это — арифметика тем медленнее, чем длиннее числа. Зато переполнения
нет, и `2 ** 1000` считается точно, в отличие от `float`.

Число бит и число единиц среди них `int` сообщает сам:


In [ ]:
(1000).bit_length(), (255).bit_count()

### Почему числа нельзя сравнивать через `is`

В лекции было сказано: `==` сравнивает значения, `is` — объекты. С целыми
числами это правило даёт неприятный сюрприз. Небольшие числа (от -5 до 256)
интерпретатор создаёт один раз при запуске и переиспользует, а остальные — нет:


In [ ]:
int("100") is int("100")

In [ ]:
int("257") is int("257")

`int("100")` дважды вернул один закешированный объект, `int("257")` — два
разных. Граница 256 — деталь реализации, она может измениться.

Вывод: **числа сравнивают только через `==`**. `is` осмысленно в одном
случае — проверка на `None`.


## 3. Деление: `//` и `%` работают не так, как в C

Целочисленное деление в Python округляет **вниз**, к минус бесконечности, а не
в сторону нуля:


In [ ]:
7 // 2, -7 // 2

В C, Java и Go `-7 / 2` дало бы `-3` — там дробная часть просто отбрасывается.
Python даёт `-4`. Разница видна и на остатке:


In [ ]:
7 % 2, -7 % 2

Остаток имеет знак делителя, поэтому при положительном делителе он всегда
неотрицателен — ровно то, что нужно для дня недели, индекса в кольцевом буфере
или часа на циферблате: `(hour + shift) % 24` не сломается на отрицательном
сдвиге.

Оба оператора связаны инвариантом:

```
(a // b) * b + (a % b) == a
```


In [ ]:
a, b = -7, 2
(a // b) * b + (a % b) == a

Когда нужны и частное, и остаток, считать их по отдельности незачем: `divmod`
возвращает пару за один раз.


In [ ]:
divmod(-7, 2), divmod(3661, 60)

Деление «вверх» отдельной функцией не оформлено, но выражается через то же
`//` двойным минусом:


In [ ]:
-(-7 // 2), -(-8 // 2)

Семь пополам — четыре занятых стола, восемь — тоже четыре. `math.ceil(7 / 2)`
даст тот же ответ, но по дороге число станет `float` и на больших значениях
потеряет точность; `-(-a // b)` остаётся целым.


## 4. `float`: почему `0.1 + 0.2` не равно `0.3`

Это не ошибка Python — так устроен любой язык с числами двойной точности.


In [ ]:
0.1 + 0.2

In [ ]:
0.1 + 0.2 == 0.3

`float` хранит число как двоичную дробь. Десятичная `0.1` в двоичной системе
периодическая — точно записать её нельзя, как нельзя записать `1/3` конечной
десятичной дробью. В памяти лежит ближайшее представимое число, и оно не равно
`0.1`. Увидеть его целиком помогает `Decimal`:


In [ ]:
from decimal import Decimal

Decimal(0.1)

При печати Python показывает кратчайшую запись, округляющуюся обратно к тому же
числу, — поэтому в выводе обычно `0.1`, а не длинный хвост.

Второе следствие: значащих бит у `float` всего 53. Как только целое перестаёт в
них помещаться, соседние целые становятся неразличимы:


In [ ]:
2.0**53 + 1 == 2.0**53

In [ ]:
2**53 + 1 == 2**53

С `int` такого не происходит — он точен всегда.

### Как тогда сравнивать

Не через `==`, а через `math.isclose`, который допускает относительную
погрешность:


In [ ]:
import math

math.isclose(0.1 + 0.2, 0.3)

Оговорка: «относительная» значит «пропорциональная величине чисел». Рядом с
нулём такой погрешности не существует, и сравнение с нулём не проходит:


In [ ]:
math.isclose(1e-9, 0.0)

Для сравнения с нулём задают абсолютный порог: `math.isclose(x, 0.0,
abs_tol=1e-9)`.

### `round` округляет не туда, куда учили в школе

`round` использует банковское округление: половина уходит к **чётному**
соседу. Так ошибка не накапливается в одну сторону при суммировании многих
значений.


In [ ]:
round(0.5), round(1.5), round(2.5), round(3.5)

Привычное «половина вверх» получается из `math.floor`:


In [ ]:
math.floor(2.5 + 0.5), math.floor(0.5 + 0.5)

И ещё одна ловушка: `round(2.675, 2)` даёт `2.67`, потому что в памяти лежит
число чуть меньше, чем `2.675`.


In [ ]:
round(2.675, 2)

### Вывод для денег

Деньги нельзя хранить во `float`. Есть два рабочих способа: считать всё в
копейках обычным `int` или брать `Decimal("0.10")` — он строится из **строки**
и хранит десятичную дробь точно. Обратите внимание: `Decimal(0.1)` из числа
уже унаследует ошибку, а `Decimal("0.1")` — нет.


In [ ]:
Decimal("0.1") + Decimal("0.2") == Decimal("0.3")

## 5. Строки: длина — это не количество букв

`len` считает не буквы, а **кодовые точки** Unicode. Номер кодовой точки
показывает `ord`, обратно переводит `chr`:


In [ ]:
ord("A"), ord("я"), chr(1071)

Проблема в том, что одна и та же буква бывает записана двумя способами. «é» —
это либо одна кодовая точка, либо «e» плюс отдельный знак ударения. Выглядят
одинаково, длина разная, `==` их не равняет:


In [ ]:
import unicodedata

composed = unicodedata.normalize("NFC", "é")
decomposed = unicodedata.normalize("NFD", "é")

len(composed), len(decomposed), composed == decomposed

Лечится приведением к одной форме — нормализацией. После неё строки
сравниваются как ожидается:


In [ ]:
unicodedata.normalize("NFC", decomposed) == composed

Похожая история с эмодзи: значок «палец вверх» — одна кодовая точка, а флаг
собран из двух региональных букв, поэтому его длина равна двум.


In [ ]:
len("👍"), len("🇷🇺")

### Длина в символах и длина в байтах

Это разные величины. Метод `encode` без аргументов кодирует строку в UTF-8, где
латинская буква занимает один байт, кириллическая — два, а эмодзи — четыре:


In [ ]:
len("привет"), len("привет".encode())

Кодировку можно указать явно. В однобайтовой cp1251 та же буква занимает один
байт вместо двух — но и записать в ней можно далеко не всякий текст:


In [ ]:
len("ы".encode()), len("ы".encode("cp1251"))

Отсюда правило: длину текста меряют по `str`, а размер файла или сообщения —
по `bytes`. Смешивать их нельзя.

### `casefold` вместо `lower`

Для сравнения текста без учёта регистра `lower` слабоват: немецкая «ß» при
`lower` остаётся собой, хотя равна «ss». `casefold` приводит и такие случаи:


In [ ]:
"ß".lower(), "ß".casefold()

In [ ]:
"STRASSE".casefold() == "straße".casefold()

Надёжное сравнение текста, введённого человеком, складывается из трёх шагов:
`strip` → `normalize` → `casefold`.


## 6. Срезы вблизи

Срез `s[start:stop:step]` в лекции уже был. Здесь — три вещи, на которых
ошибаются чаще всего.

**Первое: срез не падает за границами.** Индексация падает, срез — нет:


In [ ]:
s = "привет"

s[100:200]

Пустая строка; `s[100]` на этом месте выдал бы ошибку. Поэтому срезом безопасно
брать «первые пять символов» у строки любой длины.

**Второе: отрицательный шаг меняет смысл границ.** При `step = -1` идём справа
налево, `start` становится правым концом, а `stop` — по-прежнему
невключительной границей:


In [ ]:
s[::-1], s[-1:0:-1]

Во втором случае потерялась «п»: граница `0` не включена. Развернуть строку
целиком можно только опустив границы — `s[::-1]`.

**Третье: `-0` — это `0`.** Отрицательные индексы считают с конца, но нуля с
конца не существует, и попытка «отрезать по краю» рушится, когда край нулевой:


In [ ]:
k = 0

s[k:-k]

Ожидалась вся строка, получилась пустая: `-0` — это `0`, и срез вышел `s[0:0]`.
Правильная запись не использует минус: `s[k:len(s) - k]`.


In [ ]:
s[k : len(s) - k]

Наконец, набор границ можно сохранить в переменную — это объект `slice`:


In [ ]:
window = slice(1, 5, 2)

s[window]